# 01 — Data Extraction and Dataset Split

**Project:** AuraVision — Crowd Age Group Majority Classifier  
**Purpose:** collect images, split the dataset, and prepare the folders needed by the training and evaluation notebooks.

This notebook keeps the original data extraction and split code intact by writing each script into its `.py` file, then running it from Colab.

## Colab setup

Run this first in Google Colab. If you uploaded the ZIP, extract it and open the notebook from the extracted project folder.

In [ ]:

!pip install -q torch torchvision pillow numpy matplotlib seaborn scikit-learn requests

import os
print("Current working directory:", os.getcwd())
print("Files here:", os.listdir("."))

## Step 1 — Save the training-data extraction script

This is the original `data_extraction.py` code in notebook form.

**Security note:** The original script contains a visible Pexels API key. Replace or regenerate it before uploading to a public GitHub repository.

In [ ]:
%%writefile data_extraction.py
import requests
import os

# CONFIGURATION 
API_KEY = '45fFNqCjN4SWAHfgFxsRGTqrx1sZVoml5EFSpOdX0y9Utxyf96cujulx'
SAVE_DIR = 'crowd_dataset'
CLASSES = {
    "Children": ["children", "children at school", "playground children", "children playing", "kindergarten", "youth sports", ],
    "Adults": ["business women", "business men", "university students", "new married couples", "adults working", "music festival adults", "adults in gym", "airport terminal"],
    "Seniors": ["seniors", "seniors walking", "retirement community", "gardening seniors", "seniors socializing"]
}
IMAGES_PER_QUERY = 40 # Adjust based on your needs

def download_images():
    headers = {'Authorization': API_KEY}
    
    for label, queries in CLASSES.items():
        # Create folder for the class
        class_path = os.path.join(SAVE_DIR, label)
        os.makedirs(class_path, exist_ok=True)
        
        for query in queries:
            url = f"https://api.pexels.com/v1/search?query={query}&per_page={IMAGES_PER_QUERY}"
            response = requests.get(url, headers=headers)
            
            if response.status_code == 200:
                data = response.json()
                for i, photo in enumerate(data['photos']):
                    img_url = photo['src']['large'] # Use 'large' or 'original'
                    img_data = requests.get(img_url).content
                    
                    # Save image
                    filename = f"{query.replace(' ', '_')}_{i}.jpg"
                    with open(os.path.join(class_path, filename), 'wb') as f:
                        f.write(img_data)
                print(f"Finished downloading '{query}' for class '{label}'")
            else:
                print(f"Error {response.status_code} for query {query}")

if __name__ == "__main__":
    download_images()


## Step 2 — Run training-data extraction

This downloads the image classes into `crowd_dataset/`.

In [ ]:

!python data_extraction.py

## Step 3 — Save the optional separate test-data extraction script

This script creates `test_dataset/` with class subfolders. Run it only if you want to collect an additional labeled test set.

In [ ]:
%%writefile test_data_extraction.py
import requests
import os

# CONFIGURATION:
API_KEY = '45fFNqCjN4SWAHfgFxsRGTqrx1sZVoml5EFSpOdX0y9Utxyf96cujulx'
SAVE_DIR = 'test_dataset'
CLASSES = {
    "Children": ["Elementary school assembly", "Family day at zoo", "Kids birthday party background", "Youth soccer match crowd"],
    "Adults": ["City commuters morning", "Airport terminal crowd", "Tech conference audience", "Nightlife street scene", "Busy food court"],
    "Seniors": ["Public park morning walkers", "Senior center ballroom", "Older people gardening together", "Pensioners traveling in tour group"]
}
IMAGES_PER_QUERY = 10 # Adjust based on your needs

def download_images():
    headers = {'Authorization': API_KEY}
    
    for label, queries in CLASSES.items():
        # Create folder for the class
        class_path = os.path.join(SAVE_DIR, label)
        os.makedirs(class_path, exist_ok=True)
        
        for query in queries:
            url = f"https://api.pexels.com/v1/search?query={query}&per_page={IMAGES_PER_QUERY}"
            response = requests.get(url, headers=headers)
            
            if response.status_code == 200:
                data = response.json()
                for i, photo in enumerate(data['photos']):
                    img_url = photo['src']['large'] # Use 'large' or 'original'
                    img_data = requests.get(img_url).content
                    
                    # Save image
                    filename = f"{query.replace(' ', '_')}_{i}.jpg"
                    with open(os.path.join(class_path, filename), 'wb') as f:
                        f.write(img_data)
                print(f"Finished downloading '{query}' for class '{label}'")
            else:
                print(f"Error {response.status_code} for query {query}")

if __name__ == "__main__":
    download_images()


In [ ]:

# Optional: run this only if you want a separate downloaded test set.
# !python test_data_extraction.py

## Step 4 — Save and run the dataset split script

This is the original `split_data.py` code. It splits `crowd_dataset/` into `split_dataset/train/` and `split_dataset/test/`.

In [ ]:
%%writefile split_data.py
import os
import shutil
import random

def split_dataset(source_dir, output_dir, split_ratio=0.8):
    # Set up the new folder structure
    for folder in ['train', 'test']:
        for class_name in os.listdir(source_dir):
            os.makedirs(os.path.join(output_dir, folder, class_name), exist_ok=True)

    classes = os.listdir(source_dir)
    
    for class_name in classes:
        class_path = os.path.join(source_dir, class_name)
        if not os.path.isdir(class_path):
            continue
            
        # Get all images and shuffle them
        images = [f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))]
        random.shuffle(images)
        
        # Calculate split index
        split_point = int(len(images) * split_ratio)
        train_images = images[:split_point]
        test_images = images[split_point:]
        
        # Helper to copy files
        def copy_files(files, subset):
            for f in files:
                src = os.path.join(class_path, f)
                dst = os.path.join(output_dir, subset, class_name, f)
                shutil.copy(src, dst)
        
        copy_files(train_images, 'train')
        copy_files(test_images, 'test')
        
        print(f"Class '{class_name}': {len(train_images)} train, {len(test_images)} test.")

# EXECUTION: 
# source_dir is your original 'crowd_dataset'
# output_dir is where the new split folders will be created
split_dataset(source_dir='crowd_dataset', output_dir='split_dataset')


In [ ]:

!python split_data.py

## Step 5 — Prepare folders for the corrected training script

The corrected `train_model.py` expects `dataset/train/` with class subfolders and `dataset/test/` containing test images directly. This cell creates that structure without changing the training code.

In [ ]:
import os
import shutil
from pathlib import Path

SUPPORTED_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def prepare_training_and_test_folders(
    split_train_dir="split_dataset/train",
    split_test_dir="split_dataset/test",
    output_train_dir="dataset/train",
    output_test_dir="dataset/test"
):
    """
    Prepares folders expected by train_model.py:
      - dataset/train remains structured with class subfolders.
      - dataset/test is flattened because train_model.py runs inference on images directly inside dataset/test.

    The structured split_dataset/test folder is kept unchanged and is used later for the confusion matrix.
    """
    split_train_dir = Path(split_train_dir)
    split_test_dir = Path(split_test_dir)
    output_train_dir = Path(output_train_dir)
    output_test_dir = Path(output_test_dir)

    if not split_train_dir.exists():
        raise FileNotFoundError(f"Missing training split folder: {split_train_dir}")
    if not split_test_dir.exists():
        raise FileNotFoundError(f"Missing labeled test split folder: {split_test_dir}")

    # Copy structured training data.
    output_train_dir.parent.mkdir(parents=True, exist_ok=True)
    if output_train_dir.exists():
        shutil.rmtree(output_train_dir)
    shutil.copytree(split_train_dir, output_train_dir)

    # Flatten test images for visual/inference scripts.
    if output_test_dir.exists():
        shutil.rmtree(output_test_dir)
    output_test_dir.mkdir(parents=True, exist_ok=True)

    copied = 0
    for class_dir in sorted(split_test_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() in SUPPORTED_EXTENSIONS:
                new_name = f"{class_dir.name}_{image_path.name}"
                shutil.copy2(image_path, output_test_dir / new_name)
                copied += 1

    print(f"Prepared {output_train_dir} for training.")
    print(f"Prepared {output_test_dir} with {copied} flattened test images for inference/visualization.")
    print(f"Kept {split_test_dir} as the labeled test folder for confusion matrix evaluation.")

prepare_training_and_test_folders()

## Output of this notebook

After running this notebook, you should have:

```text
split_dataset/train/     # labeled train split
split_dataset/test/      # labeled test split for confusion matrix
dataset/train/           # training folder used by train_model.py
dataset/test/            # flattened test images for inference and visualization
```